# 11 - 可复用 PINN 模板

这个 notebook 是给你改自己问题用的模板。默认仍然求解一维 Poisson 方程，但代码被整理成几个清晰函数：

- `build_model`：创建网络
- `pde_residual`：写方程残差
- `boundary_loss`：写边界条件
- `train`：训练循环
- `evaluate`：画图和误差评估

迁移到新问题时，通常只需要改残差、边界/初值条件和输入维度。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(11)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
def build_model(in_dim=1, out_dim=1, width=48, depth=4):
    layers = [nn.Linear(in_dim, width), nn.Tanh()]
    for _ in range(depth - 1):
        layers += [nn.Linear(width, width), nn.Tanh()]
    layers += [nn.Linear(width, out_dim)]
    return nn.Sequential(*layers).to(device)

def exact_solution(x):
    return torch.sin(math.pi * x)

def pde_residual(model, x):
    x.requires_grad_(True)
    u = model(x)
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    forcing = -(math.pi**2) * torch.sin(math.pi * x)
    return u_xx - forcing

def boundary_loss(model):
    xb = torch.tensor([[0.0], [1.0]], device=device)
    ub = torch.zeros_like(xb)
    return torch.mean((model(xb) - ub)**2)

In [ ]:
def train(model, epochs=6000, n_f=256, lr=1e-3, bc_weight=50.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        x_f = torch.rand(n_f, 1, device=device)
        loss_phys = torch.mean(pde_residual(model, x_f)**2)
        loss_bc = boundary_loss(model)
        loss = loss_phys + bc_weight * loss_bc
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append([loss.item(), loss_phys.item(), loss_bc.item()])
        if epoch % 1000 == 0:
            print(f"epoch={epoch:4d}, loss={loss.item():.3e}, phys={loss_phys.item():.3e}, bc={loss_bc.item():.3e}")
    return history

def evaluate(model, history):
    x = torch.linspace(0, 1, 400, device=device).reshape(-1, 1)
    with torch.no_grad():
        pred = model(x)
        exact = exact_solution(x)
    h = torch.tensor(history)
    plt.figure(figsize=(11, 4))
    plt.subplot(1, 2, 1)
    plt.plot(x.cpu(), exact.cpu(), label="exact")
    plt.plot(x.cpu(), pred.cpu(), "--", label="PINN")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.semilogy(h[:, 0], label="total")
    plt.semilogy(h[:, 1], label="physics")
    plt.semilogy(h[:, 2], label="boundary")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("max abs error=", torch.max(torch.abs(pred - exact)).item())

In [ ]:
model = build_model()
history = train(model)
evaluate(model, history)

改模板建议：

1. 如果是时间相关问题，把输入维度改成 2，例如 `(x,t)`。
2. 在 `pde_residual` 里增加需要的导数。
3. 把边界条件、初值条件分别写成函数，训练时加权求和。
4. 每次只改一个因素，方便定位训练失败原因。